# Neural Operator Tutorial: Learning Maps Between Function Spaces

This notebook studies the neural-operator viewpoint introduced by Kovachki et al., *Neural Operator: Learning Maps Between Function Spaces* (JMLR 2023). The goal is not just to run a model, but to connect:

- the **operator-learning problem** $G^\dagger : \mathcal{A} \to \mathcal{U}$
- the **discretization-invariant neural operator architecture**
- the **Fourier neural operator (FNO)** specialization
- an **exactly solvable PDE family** that lets us verify the data-generating operator
- **resolution transfer** from a coarse training grid to a finer evaluation grid

Primary reference:

- Kovachki et al. (2023), *Neural Operator: Learning Maps Between Function Spaces*, JMLR 24:1-97
- PDF: https://jmlr.org/papers/volume24/21-1524/21-1524.pdf


## 1 — Theory: What Is a Neural Operator?

The paper formulates operator learning as the task of approximating a nonlinear map

$$
G^\dagger : \mathcal{A} \to \mathcal{U},
$$

where both the input and the output live in infinite-dimensional function spaces. Instead of learning a mapping between fixed-size vectors, we learn a map between **functions**.

The paper emphasizes three core ideas:

1. **Discretization invariance**: the same learned parameters should work across different discretizations.
2. **Neural operator layers**: each layer is itself an operator, not just a matrix acting on a fixed-size vector.
3. **Universal approximation of continuous operators**: the neural-operator family can approximate continuous nonlinear operators on compact sets.

At a high level, the architecture is:

$$
v_0(x) = P(a(x)), \qquad
v_{t+1}(x) = \sigma\left(W_t v_t(x) + \int \kappa_t(x, y) v_t(y)\, d\nu(y) + b_t(x)\right), \qquad
u(x) = Q(v_T(x)).
$$

Here:

- $P$ is a pointwise **lifting** map into a higher-dimensional channel space
- each hidden layer mixes a **local linear path** and a **nonlocal integral operator**
- $Q$ is a pointwise **projection** back to the target function space

In the paper's terminology, this operator-level construction is what allows the model to remain meaningful as the grid changes.


## 2 — Theory: Why Fourier Neural Operators Matter

The Fourier neural operator (FNO) is a practical neural-operator instantiation in which the nonlocal integral operator is parameterized in Fourier space:

1. transform the hidden state into Fourier coefficients
2. apply a learnable linear map to the **low-frequency modes**
3. invert the transform back to physical space
4. add a local linear path and a nonlinearity

This matters for two reasons:

- it reduces operator application cost to roughly **$O(n \log n)$**
- the same learned low-mode parameters can be reused at multiple resolutions, enabling **zero-shot super-resolution**

The paper also emphasizes that FNOs are not restricted to periodic problems in practice, because the local linear path and nonlinear decoder help recover non-periodic structure. That is why this notebook includes a non-periodic Dirichlet boundary-value problem.


## 3 — PDE Family Used in This Notebook

We study the 1-D variable-coefficient diffusion problem

$$
-\big(a(x) u'(x)\big)' = f(x), \qquad x \in (0,1), \qquad u(0)=u(1)=0,
$$

where both:

- the diffusivity $a(x)$ is a random positive function
- the forcing $f(x)$ is a random smooth function

This gives us a true operator

$$
(a, f) \mapsto u.
$$

The reason this family is excellent for an operator-learning tutorial is that the solution operator is **nonlocal** but still admits an exact 1-D formula. Let

$$
F(x) = \int_0^x f(s)\, ds.
$$

Then

$$
a(x)u'(x) = C - F(x),
$$

for some constant $C$. Integrating once more and enforcing $u(1)=0$ yields

$$
C =
\frac{\int_0^1 F(s) / a(s)\, ds}{\int_0^1 1 / a(s)\, ds},
\qquad
u(x) = \int_0^x \frac{C - F(s)}{a(s)}\, ds.
$$

So the training data is not generated by an approximate black-box solver. It is produced by a **verified exact operator formula** evaluated numerically on each grid.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from physics_informed_neural_network.neural_operator.data import (
    build_dataset_splits,
    compute_discrete_diffusion_residual,
)
from physics_informed_neural_network.neural_operator.model import FourierNeuralOperator1d
from physics_informed_neural_network.neural_operator.pipeline import run_neural_operator_experiment
from physics_informed_neural_network.neural_operator.plotting import (
    apply_plot_style,
    plot_dataset_examples,
    plot_frequency_spectrum,
    plot_prediction_comparison,
    plot_resolution_metrics,
    plot_training_history,
)
from physics_informed_neural_network.neural_operator.presets import build_smoke_test_config, build_tutorial_config

apply_plot_style()
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

## 4 — Pydantic Configuration

The tutorial code is structured like library code, not notebook-only code:

- Pydantic config models define the problem, dataset, model, and optimizer
- reusable functions generate exact data at arbitrary resolution
- the FNO model supports variable grid lengths
- the trainer owns normalization, optimization, evaluation, and prediction

Use the tutorial preset for a meaningful run, or swap to `build_smoke_test_config()` when you want a very fast sanity check.


In [ ]:
config = build_tutorial_config(output_dir=PROJECT_ROOT / "artifacts" / "notebook_neural_operator")
display(config.model_dump())

In [ ]:
splits = build_dataset_splits(config)
dataset_frame = pd.DataFrame(
    [
        {"split": "train", "samples": splits.train.n_samples, "resolution": splits.train.resolution},
        {"split": "validation", "samples": splits.validation.n_samples, "resolution": splits.validation.resolution},
        {"split": "test", "samples": splits.test.n_samples, "resolution": splits.test.resolution},
        {"split": "refined_test", "samples": splits.refined_test.n_samples, "resolution": splits.refined_test.resolution},
    ]
)
display(dataset_frame)

In [ ]:
sample = splits.train.sample(0)
exact_residual = compute_discrete_diffusion_residual(
    sample.grid,
    sample.diffusion,
    sample.solution,
    sample.forcing,
)
verification_frame = pd.DataFrame(
    [
        {
            "u(0)": sample.solution[0],
            "u(1)": sample.solution[-1],
            "mean_abs_residual": np.mean(np.abs(exact_residual)),
            "max_abs_residual": np.max(np.abs(exact_residual)),
        }
    ]
)
display(verification_frame)

In [ ]:
fig_examples = plot_dataset_examples(splits.train)
fig_examples

## 5 — FNO Architecture

The model below is intentionally small enough to understand directly:

- a pointwise linear **lift** from input channels `[a(x), f(x), x]`
- repeated Fourier residual blocks with
  - low-mode spectral convolution
  - a local $1 \times 1$ linear path
  - a nonlinearity
- a pointwise **projection** back to the scalar solution channel

Because the spectral weights act on Fourier modes rather than on a fixed flattened vector length, the same trained model can be evaluated at a different number of grid points.


In [ ]:
model = FourierNeuralOperator1d(config.model)
print(model.architecture_string())
print(f"Trainable parameters: {model.count_parameters():,}")
model

## 6 — Train the Neural Operator

The tutorial preset trains on a coarse grid and later evaluates on both:

- the same resolution used for training
- a finer grid the model never saw during training

That second evaluation is the notebook's practical test of discretization transfer.


In [ ]:
experiment = run_neural_operator_experiment(config)

In [ ]:
evaluation_frame = pd.DataFrame(
    [
        {
            "split": evaluation.split,
            "resolution": evaluation.resolution,
            **evaluation.metrics.model_dump(),
        }
        for evaluation in experiment.summary.evaluations.values()
    ]
)
display(evaluation_frame)
display(experiment.summary.model_dump())

In [ ]:
history_frame = experiment.history.to_frame()
display(history_frame.tail())
fig_history = plot_training_history(experiment.history)
fig_history

## 7 — Native-Resolution Prediction

This is the easier test case: evaluation on the same resolution seen during training.


In [ ]:
native_index = 0
native_sample = experiment.datasets.test.sample(native_index)
fig_native = plot_prediction_comparison(
    native_sample,
    experiment.native_prediction[native_index],
    title="Native-resolution test prediction",
)
fig_native

## 8 — Zero-Shot Resolution Transfer

This is the more interesting test: we evaluate the same latent test functions on a finer grid without retraining the model. This mirrors the paper's resolution-transfer story in a smaller 1-D setting.


In [ ]:
refined_sample = experiment.datasets.refined_test.sample(native_index)
fig_refined = plot_prediction_comparison(
    refined_sample,
    experiment.refined_prediction[native_index],
    title="Refined-grid zero-shot transfer prediction",
)
fig_refined

In [ ]:
fig_metrics = plot_resolution_metrics(experiment.summary)
fig_metrics

In [ ]:
def per_sample_relative_l2(prediction: np.ndarray, target: np.ndarray) -> np.ndarray:
    numerator = np.linalg.norm(prediction - target, axis=1)
    denominator = np.linalg.norm(target, axis=1) + 1e-12
    return numerator / denominator


native_relative = per_sample_relative_l2(experiment.native_prediction, experiment.datasets.test.solution)
refined_relative = per_sample_relative_l2(experiment.refined_prediction, experiment.datasets.refined_test.solution)
display(
    pd.DataFrame(
        {
            "native_relative_l2": native_relative,
            "refined_relative_l2": refined_relative,
        }
    ).describe()
)

## 9 — Spectral Analysis

One reason FNOs work well is that many PDE solution operators are dominated by low-to-mid frequency structure even when the final output contains higher-frequency details. The spectral plot below checks how well the learned operator reproduces the target energy across modes.


In [ ]:
fig_spectrum = plot_frequency_spectrum(
    refined_sample.grid,
    refined_sample.solution,
    experiment.refined_prediction[native_index],
    title="Refined-grid spectrum: exact vs prediction",
)
fig_spectrum

In [ ]:
exact_residual = compute_discrete_diffusion_residual(
    refined_sample.grid,
    refined_sample.diffusion,
    refined_sample.solution,
    refined_sample.forcing,
)
predicted_residual = compute_discrete_diffusion_residual(
    refined_sample.grid,
    refined_sample.diffusion,
    experiment.refined_prediction[native_index],
    refined_sample.forcing,
)
display(
    pd.DataFrame(
        [
            {
                "quantity": "exact solution",
                "mean_abs_residual": np.mean(np.abs(exact_residual)),
                "max_abs_residual": np.max(np.abs(exact_residual)),
            },
            {
                "quantity": "FNO prediction",
                "mean_abs_residual": np.mean(np.abs(predicted_residual)),
                "max_abs_residual": np.max(np.abs(predicted_residual)),
            },
        ]
    )
)

## 10 — Interpretation

What this notebook demonstrates:

- **Operator learning**: the model maps function-valued inputs $(a, f)$ to a function-valued output $u$.
- **Discretization awareness**: the same model parameters were reused on a finer grid.
- **Exact supervision**: the training targets come from an analytically derived operator formula.
- **Nonlocality**: changing $a(x)$ anywhere in the domain can change the integration constant $C$, and therefore the entire solution.
- **FNO intuition**: low Fourier modes carry global operator structure efficiently, while the local path and nonlinear decoder restore detail and handle non-periodic behavior.

Important limitations:

- this notebook explains the key theory, but it does **not reproduce the formal proofs** of universal approximation or discretization invariance from the paper
- the example is 1-D and uniform-grid, while the full neural-operator framework also covers broader operator classes and alternative parameterizations
- strong performance here does not automatically imply the same data efficiency on harder PDE families

For a faster sanity check, replace `build_tutorial_config()` with `build_smoke_test_config()` and rerun the notebook.


---

# Part II — 2-D Darcy Flow

The second half of this notebook extends the operator-learning story to **two spatial dimensions** using the classic Darcy-flow problem:

$$
-\nabla \cdot \big(a(x,y)\,\nabla u(x,y)\big) = f(x,y), \quad (x,y) \in (0,1)^2, \quad u\big|_{\partial\Omega} = 0.
$$

This problem is widely used as a benchmark in the neural-operator literature because:

- it involves a **heterogeneous** 2-D diffusivity field $a(x,y)$
- the solution $u(x,y)$ depends **nonlocally** on $a$ through the elliptic inverse operator
- it tests whether the FNO architecture generalizes from 1-D to 2-D

We use the FNO-2d variant with 2-D spectral convolutions, coordinate augmentation, and the same lift → spectral blocks → project architecture.


In [ ]:
from physics_informed_neural_network.neural_operator.pipeline_2d import run_darcy_experiment
from physics_informed_neural_network.neural_operator.data_2d import build_darcy_splits
from physics_informed_neural_network.neural_operator.plotting_2d import (
    plot_darcy_3d_surface,
    plot_darcy_cross_sections,
    plot_darcy_dataset_examples,
    plot_darcy_error_distribution,
    plot_darcy_prediction_comparison,
    plot_darcy_resolution_metrics,
)
from physics_informed_neural_network.neural_operator.presets import (
    build_darcy_tutorial_config,
    build_darcy_smoke_test_config,
)

## 11 — 2-D Darcy Configuration and Data

The 2-D Darcy pipeline mirrors the 1-D stack: Pydantic configs define the problem, a GRF sampler generates
random diffusivity fields, a finite-difference solver produces exact solutions, and the FNO-2d model
learns the $(a, f) \mapsto u$ operator.


In [ ]:
darcy_config = build_darcy_tutorial_config(
    output_dir=PROJECT_ROOT / "artifacts" / "notebook_darcy"
)
display(darcy_config.model_dump())

In [ ]:
darcy_splits = build_darcy_splits(darcy_config)
darcy_dataset_frame = pd.DataFrame([
    {"split": "train", "samples": darcy_splits.train.n_samples, "resolution": darcy_splits.train.resolution},
    {"split": "validation", "samples": darcy_splits.validation.n_samples, "resolution": darcy_splits.validation.resolution},
    {"split": "test", "samples": darcy_splits.test.n_samples, "resolution": darcy_splits.test.resolution},
    {"split": "refined_test", "samples": darcy_splits.refined_test.n_samples, "resolution": darcy_splits.refined_test.resolution},
])
display(darcy_dataset_frame)

In [ ]:
fig_darcy_examples = plot_darcy_dataset_examples(darcy_splits.train, sample_indices=(0, 1, 2))
fig_darcy_examples

## 12 — Train the 2-D FNO

In [ ]:
darcy_experiment = run_darcy_experiment(darcy_config)

In [ ]:
darcy_summary = pd.DataFrame([darcy_experiment.summary.model_dump()])
display(darcy_summary)

## 13 — 2-D Prediction Quality

The key figure: diffusivity input → exact solution → FNO prediction → absolute error map (log scale).


In [ ]:
darcy_test_sample = darcy_splits.test.sample(0)
fig_darcy_pred = plot_darcy_prediction_comparison(
    darcy_test_sample,
    darcy_experiment.native_prediction[0],
    title="2-D Darcy: FNO prediction vs exact solution",
)
fig_darcy_pred

In [ ]:
fig_darcy_3d = plot_darcy_3d_surface(
    darcy_test_sample,
    darcy_experiment.native_prediction[0],
    title="2-D Darcy — 3-D surface comparison",
)
fig_darcy_3d

In [ ]:
fig_darcy_cross = plot_darcy_cross_sections(
    darcy_test_sample,
    darcy_experiment.native_prediction[0],
)
fig_darcy_cross

## 14 — 2-D Error Distribution and Resolution Transfer

In [ ]:
fig_darcy_error = plot_darcy_error_distribution(
    darcy_experiment.native_prediction,
    darcy_splits.test.solution,
    title="2-D Darcy: per-sample relative $L^2$ error distribution",
)
fig_darcy_error

In [ ]:
fig_darcy_res = plot_darcy_resolution_metrics(darcy_experiment.summary)
fig_darcy_res

## 15 — Summary

This notebook demonstrated operator learning across both 1-D and 2-D problems:

- **1-D diffusion**: exact operator formula, FNO-1d, resolution transfer, spectral analysis
- **2-D Darcy flow**: GRF-based data, FNO-2d, 3-D surface visualization, cross-section comparison

The same architectural principles — Fourier-space mixing, local linear paths, discretization-aware design — apply to both cases, and the library code is structured to make that reuse clean and testable.
